# CROMA + BEN-GE-800 Phase 2B Sensor Fairness

Phase 2B uses real paired Sentinel-1/Sentinel-2 BEN-GE-800 samples to compare CROMA SAR-only, optical-only, and S1+S2 fusion. This notebook downloads/prepares BEN-GE-800 directly in Colab local storage and does not use Google Drive staging.


In [ ]:
# 1. Clone or update this repo
REPO_URL = "https://github.com/strivekboy-coder/rsfm-fairness-audit.git"
REPO_DIR = "rsfm-fairness-audit"

from pathlib import Path
if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}


In [ ]:
# 2. Install package and CROMA/BEN-GE dependencies
!python -m pip install -e .
!python -m pip install -r requirements-croma.txt


In [ ]:
# 3. Check GPU
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# 4. Configure CROMA official implementation path
from pathlib import Path
import yaml

CROMA_REPO_DIR = "/content/CROMA"
if Path(CROMA_REPO_DIR).exists():
    !git -C {CROMA_REPO_DIR} pull
else:
    !git clone https://github.com/antofuller/CROMA {CROMA_REPO_DIR}

for config_name in ["croma_sar.yaml", "croma_optical_benge.yaml", "croma_both.yaml"]:
    path = Path("configs/models") / config_name
    config = yaml.safe_load(path.read_text())
    config["repo_path"] = CROMA_REPO_DIR
    config["source_file_path"] = None
    config["checkpoint_path"] = None
    config["allow_hf_download"] = True
    config["device"] = "auto"
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    print("---", path)
    print(path.read_text())


In [ ]:
# 5. Download and prepare BEN-GE-800 paired S1/S2 subset
DATA_ROOT = "data/ben_ge_800_subset64"
CACHE_DIR = "data/_cache/ben_ge_800"
SAR_OUTPUT = "outputs/croma_benge800_sar64"
OPTICAL_OUTPUT = "outputs/croma_benge800_optical64"
BOTH_OUTPUT = "outputs/croma_benge800_both64"
COMPARISON_OUTPUT = "outputs/comparisons/croma_benge800_sensor64"

!python scripts/prepare_ben_ge_800_subset.py \
  --output-dir {DATA_ROOT} \
  --cache-dir {CACHE_DIR} \
  --max-samples 64 \
  --seed 42
!head -5 {DATA_ROOT}/metadata.csv


In [ ]:
# 6. Preflight checks
!python -m rsfm_fairness_audit.cli check-real --dataset ben_ge --model croma --model-config configs/models/croma_sar.yaml --data-root {DATA_ROOT} --sensor-mode S1
!python -m rsfm_fairness_audit.cli check-real --dataset ben_ge --model croma --model-config configs/models/croma_optical_benge.yaml --data-root {DATA_ROOT} --sensor-mode S2
!python -m rsfm_fairness_audit.cli check-real --dataset ben_ge --model croma --model-config configs/models/croma_both.yaml --data-root {DATA_ROOT} --sensor-mode S1+S2


In [ ]:
# 7. Run CROMA SAR / Optical / Both 64-sample experiments
!python -m rsfm_fairness_audit.cli run-real --dataset ben_ge --dataset-root {DATA_ROOT} --model croma --config configs/models/croma_sar.yaml --sensor-mode S1 --output-dir {SAR_OUTPUT} --max-samples 64 --chunk-size 64 --streaming-embeddings true
!python -m rsfm_fairness_audit.cli run-real --dataset ben_ge --dataset-root {DATA_ROOT} --model croma --config configs/models/croma_optical_benge.yaml --sensor-mode S2 --output-dir {OPTICAL_OUTPUT} --max-samples 64 --chunk-size 64 --streaming-embeddings true
!python -m rsfm_fairness_audit.cli run-real --dataset ben_ge --dataset-root {DATA_ROOT} --model croma --config configs/models/croma_both.yaml --sensor-mode S1+S2 --output-dir {BOTH_OUTPUT} --max-samples 64 --chunk-size 64 --streaming-embeddings true


In [ ]:
# 8. Compare sensor modes and inspect outputs
!python -m rsfm_fairness_audit.cli compare-sensor-modes \
  --dataset ben_ge \
  --run sar={SAR_OUTPUT} \
  --run optical={OPTICAL_OUTPUT} \
  --run both={BOTH_OUTPUT} \
  --output-dir {COMPARISON_OUTPUT}

!find {COMPARISON_OUTPUT} -maxdepth 3 -type f -print
!sed -n '1,160p' {COMPARISON_OUTPUT}/report.md

from IPython.display import Image, display
for fig in [f"{COMPARISON_OUTPUT}/figures/sensor_fairness_heatmap.png", f"{COMPARISON_OUTPUT}/figures/average_vs_worst_sensor_mode.png"]:
    if Path(fig).exists():
        display(Image(filename=fig))


## BWER Sensor-Mode Audit

This final section runs the new BWER audit on the existing SAR, optical, and both CROMA outputs. It uses the completed `predictions.csv` files and does not rerun model inference.


In [ ]:
# 9. Run BWER audit on existing CROMA BEN-GE sensor-mode outputs
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

BWER_OUTPUT = "outputs/audit/croma_benge800_sensor64"
BWER_INPUT = f"{BWER_OUTPUT}/sensor_mode_audit_table.csv"
Path(BWER_OUTPUT).mkdir(parents=True, exist_ok=True)

runs = {
    "sar": SAR_OUTPUT,
    "optical": OPTICAL_OUTPUT,
    "both": BOTH_OUTPUT,
}
frames = []
for sensor_mode, run_dir in runs.items():
    predictions_path = Path(run_dir) / "predictions.csv"
    if not predictions_path.exists():
        print(f"Skipping {sensor_mode}: missing {predictions_path}")
        continue
    frame = pd.read_csv(predictions_path)
    frame["dataset"] = "ben_ge_800"
    frame["model"] = "croma"
    frame["task"] = "classification"
    frame["split"] = frame.get("split", "all")
    frame["sensor_mode"] = sensor_mode
    frame["class_label"] = frame.get("class_label", frame["label"].astype(str))
    frame["unit_id"] = sensor_mode + "__" + frame["sample_id"].astype(str)
    frame["score"] = frame.get("score", frame.get("correct"))
    frames.append(frame)

if not frames:
    raise RuntimeError("No existing CROMA BEN-GE predictions.csv files were found. Run the SAR/optical/both cells first.")
audit_table = pd.concat(frames, ignore_index=True)
audit_table.to_csv(BWER_INPUT, index=False)
print(f"Wrote {len(audit_table)} audit rows to {BWER_INPUT}")

!python -m rsfm_fairness_audit.cli evaluate-bwer \
  --audit-table {BWER_INPUT} \
  --dataset ben_ge_800 \
  --model croma \
  --task classification \
  --slice-variable sensor_mode \
  --balance-variable class_label \
  --output-dir {BWER_OUTPUT} \
  --missing-balance-policy renormalize \
  --bootstrap 50

for csv_name in ["bwer_summary.csv", "bwer_by_slice.csv", "support_diagnostics.csv"]:
    csv_path = Path(BWER_OUTPUT) / csv_name
    print(f"\n=== {csv_name} ===")
    if csv_path.exists() and csv_path.stat().st_size > 0:
        display(pd.read_csv(csv_path).head(20))
    else:
        print("missing or empty")

for fig_name in ["average_vs_bwer.png", "raw_vs_balanced_bwer.png", "worst_tail_slices.png", "slice_risk_heatmap.png"]:
    fig_path = Path(BWER_OUTPUT) / "figures" / fig_name
    if fig_path.exists():
        print(fig_name)
        display(Image(filename=str(fig_path)))


In [ ]:
# 10. Package final selected Phase 2B artifacts only
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

FINAL_DIR = Path(COMPARISON_OUTPUT)
ZIP_PATH = Path("croma_benge800_phase2b_sensor64_results.zip")
files_to_zip = []
for name in ["report.md"]:
    path = FINAL_DIR / name
    if path.exists():
        files_to_zip.append(path)
for folder_name in ["tables", "figures"]:
    folder = FINAL_DIR / folder_name
    if folder.exists():
        files_to_zip.extend(sorted(p for p in folder.rglob("*") if p.is_file()))
with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for path in files_to_zip:
        archive.write(path, path.relative_to(FINAL_DIR).as_posix())
print(f"Packaged {len(files_to_zip)} artifacts into {ZIP_PATH}")
from google.colab import files
files.download(str(ZIP_PATH))


## Optional Cleanup

Uncomment only after the final zip has downloaded successfully.


In [ ]:
# !rm -rf data/ben_ge_800_subset64
# !rm -rf data/_cache/ben_ge_800
# !rm -rf outputs/croma_benge800_sar64 outputs/croma_benge800_optical64 outputs/croma_benge800_both64
# !rm -rf outputs/comparisons/croma_benge800_sensor64
